# MoodLens — Fine-tune BERT on Ekman-6 (Colab)

Self-contained: no repo clone needed. Mirrors `backend/ml/train.py` + `data.py`.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Produces `ekman-bert.zip`. Download it, unzip into `backend/models/ekman-bert/`,
and set `MODEL_NAME=models/ekman-bert` in `backend/.env`. Total time ~20–40 min.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

## 2. Install dependencies (pinned to match the project)

In [ ]:
!pip install -q transformers==4.47.1 datasets==3.2.0 scikit-learn==1.6.0 accelerate==1.2.1

## 3. Ekman label space + GoEmotions→Ekman mapping
Same mapping as `app/core/emotions.py` — one source of truth, copied here so the
notebook runs standalone.

In [ ]:
EKMAN = ["joy", "anger", "sadness", "fear", "surprise", "disgust", "neutral"]
EKMAN_LABEL2ID = {e: i for i, e in enumerate(EKMAN)}
EKMAN_ID2LABEL = {i: e for e, i in EKMAN_LABEL2ID.items()}

GOEMOTIONS_TO_EKMAN = {
    "amusement": "joy", "excitement": "joy", "joy": "joy", "love": "joy",
    "desire": "joy", "optimism": "joy", "caring": "joy", "pride": "joy",
    "admiration": "joy", "gratitude": "joy", "relief": "joy", "approval": "joy",
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    "sadness": "sadness", "disappointment": "sadness", "embarrassment": "sadness",
    "grief": "sadness", "remorse": "sadness",
    "fear": "fear", "nervousness": "fear",
    "surprise": "surprise", "realization": "surprise", "confusion": "surprise",
    "curiosity": "surprise",
    "disgust": "disgust",
    "neutral": "neutral",
}

## 4. Load GoEmotions, collapse to single-label Ekman
Keep only rows that map to exactly one Ekman emotion (documented simplification).

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

raw = load_dataset("go_emotions", "simplified")
go_names = raw["train"].features["labels"].feature.names

def convert(split):
    texts, labels = [], []
    for ex in split:
        buckets = {GOEMOTIONS_TO_EKMAN.get(go_names[i]) for i in ex["labels"]}
        buckets.discard(None)
        if len(buckets) != 1:
            continue
        emo = next(iter(buckets))
        texts.append(ex["text"]) ; labels.append(EKMAN_LABEL2ID[emo])
    return Dataset.from_dict({"text": texts, "label": labels})

ds = DatasetDict({s: convert(raw[s]) for s in raw})

for s, d in ds.items():
    counts = {e: 0 for e in EKMAN}
    for l in d["label"]:
        counts[EKMAN[l]] += 1
    print(f"[{s}] {len(d)} rows -> {counts}")

## 5. Tokenise, build model, fine-tune (3 epochs)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding)

BASE = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(BASE)
ds_tok = ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=len(EKMAN),
    id2label=EKMAN_ID2LABEL, label2id=EKMAN_LABEL2ID)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro", zero_division=0)}

args = TrainingArguments(
    output_dir="ckpt", num_train_epochs=3,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=2e-5, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    logging_steps=100, report_to="none")

trainer = Trainer(model=model, args=args,
    train_dataset=ds_tok["train"], eval_dataset=ds_tok["validation"],
    tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics)

trainer.train()

## 6. Final test-set score (compare this to the 0.663 baseline macro-F1)

In [ ]:
print(trainer.evaluate(ds_tok["test"]))

## 7. Save + download the model
Unzip into `backend/models/ekman-bert/`, then set `MODEL_NAME=models/ekman-bert`.

In [ ]:
trainer.save_model("ekman-bert")
tok.save_pretrained("ekman-bert")
!zip -r -q ekman-bert.zip ekman-bert
from google.colab import files
files.download("ekman-bert.zip")